# ĐỒ ÁN GIỮA KỲ: HỌC SÂU NÂNG CAO (ADVANCED DEEP LEARNING)
## Đề tài: Deep Learning-Based Voice Command Recognition for Mobile Robots
**Giảng viên hướng dẫn:** TS. Lê Minh Huy (ICSLab, Trường Đại học Phenikaa)
**Sinh viên thực hiện:** Nguyễn Hải Dương

---
### Tóm tắt nội dung báo cáo nghiên cứu:
1. **Bối cảnh & Dữ liệu:** Nhận diện 7 khẩu lệnh tiếng Việt điều khiển robot di động (KhoiDong, KhongXacDinh, Lui, RePhai, ReTrai, TamDung, Tien). Thu thập thực tế 700 file WAV (16,000 Hz).
2. **Cải tiến Tiền xử lý SOTA:** Chuẩn hóa khung thời gian cố định 2.0s (Center-Pad/Crop), trích xuất Mel-Spectrogram (128x63) và tiền xử lý Offline lưu ra tensor (tăng tốc độ nạp dữ liệu hơn 100 lần).
3. **Kỹ thuật Tăng cường Dữ liệu:** Ứng dụng **SpecAugment** (Google Brain) trực tiếp trên dải tần số (Frequency Masking) và thời gian (Time Masking).
4. **Kiến trúc mạng SOTA:** Đề xuất VoiceResNet và SE_VoiceResNet tích hợp đường tắt Residual (F(x) + x), Channel Attention (Squeeze-and-Excitation) và Global Average Pooling (GAP) giúp giảm đến **84% số tham số** so với kiến trúc CNN thô môn cơ bản.
5. **Thực nghiệm Khoa học:**
   * **Ablation Study (5 models M1 -> M5):** Chứng minh vai trò bóc tách của Dropout, BatchNorm, Skip Connection và SE Attention.
   * **Optimizer Benchmark:** So sánh thực nghiệm giữa SGD, SGD Momentum, Adam và AdamW.
   * **Trade-off Analysis:** Đo đạc Parameters, FLOPs, Latency (ms/sample) và vẽ đường biên Pareto.
   * **Error Analysis & XAI:** Phân tích ma trận nhầm lẫn (Confusion Matrix) và bản đồ nhiệt **Grad-CAM**.


### 1. Khởi tạo Môi trường & Nạp Dữ liệu Tiền xử lý


In [ ]:
import os, sys, json
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

sys.path.append(".")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

data = torch.load("dataset_processed.pt", weights_only=False)
print(f"Classes: {data['class_names']}")
print(f"Train samples: {data['X_train'].shape[0]} | Val: {data['X_val'].shape[0]} | Test: {data['X_test'].shape[0]}")
print(f"Mel-Spectrogram shape: {data['mel_shape']}")


from dataset import SpecAugment
spec_aug = SpecAugment(freq_mask_param=16, time_mask_param=12, p=1.0)
sample_spec = data["X_train"][0]
augmented_spec = spec_aug(sample_spec.clone())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
im0 = axes[0].imshow(sample_spec[0].numpy(), aspect="auto", origin="lower", cmap="magma")
axes[0].set_title("(a) Original Mel-Spectrogram (2.0s)", fontsize=12, fontweight="bold", pad=10)
axes[0].set_xlabel("Time Frames", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Mel Frequency Bins", fontsize=11, fontweight="bold")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(augmented_spec[0].numpy(), aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("(b) SpecAugment (Frequency & Time Masking)", fontsize=12, fontweight="bold", pad=10)
axes[1].set_xlabel("Time Frames", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Mel Frequency Bins", fontsize=11, fontweight="bold")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()


In [ ]:
from dataset import SpecAugment
spec_aug = SpecAugment(freq_mask_param=16, time_mask_param=12)
sample_spec = data["X_train"][0]
augmented_spec = spec_aug(sample_spec.clone())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
im0 = axes[0].imshow(sample_spec[0].numpy(), aspect="auto", origin="lower", cmap="magma")
axes[0].set_title("Mel-Spectrogram Chuan hoa (2.0s)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Time Frames")
axes[0].set_ylabel("Mel Frequency Bins")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(augmented_spec[0].numpy(), aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("Mel-Spectrogram sau SpecAugment (Freq & Time Masking)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Time Frames")
axes[1].set_ylabel("Mel Frequency Bins")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()


### 3. Kết quả Thực nghiệm Nghiên cứu Bóc tách (Ablation Study)
Bảng tổng hợp chỉ số kỹ thuật của 5 mô hình thực nghiệm (M1 -> M5):


In [ ]:
with open("results/ablation_summary.json") as f:
    ablation_data = json.load(f)

import pandas as pd
df = pd.DataFrame(ablation_data)
df.columns = ["Model Name", "Params", "FLOPs (M)", "Latency (ms)", "Val Acc (%)", "Test Acc (%)", "Train Time (s)"]
display(df)


### 4. Phân tích Đánh đổi Hiệu năng (Pareto Trade-off Analysis)


In [ ]:
display(Image(filename="results/pareto_tradeoff.png"))


### 5. Thực nghiệm So sánh các Thuật toán Tối ưu (Optimizer Benchmark)


In [ ]:
display(Image(filename="results/optimizer_comparison.png"))


### 6. Phân tích Lỗi (Error Analysis - Confusion Matrix)


In [ ]:
display(Image(filename="results/confusion_matrix_M5.png"))


### 7. Trực quan hóa Mở hộp đen (Explainable AI - Grad-CAM)


In [ ]:
import glob
cam_images = sorted(glob.glob("results/gradcam_sample_*.png"))
for img_path in cam_images:
    display(Image(filename=img_path))


### 8. Kết luận & Định hướng Đồ án Cuối kỳ (Final Project Roadmap)
* **Thành tựu:** Hoàn thiện pipeline tiền xử lý SOTA, kiến trúc VoiceResNet & SE_VoiceResNet giảm 84% tham số (còn 310K), độ trễ suy luận chỉ ~5.3ms/mẫu.
* **Kế hoạch Cuối kỳ:** Xuất ONNX/TFLite và tích hợp micro nhận diện thời gian thực trên robot di động.
